# Modelagem probabilística e teoria de filas (SimulaçãoFilaMG1Aula.ipynb)

- **Repositório:** `analise-de-dados-fundamentos`  
- **Tecnologias:** SciPy, NumPy  
- **Arquivo Original:** `SimulaçãoFilaMG1Aula.ipynb`  

---


# Simulação de filas MG1

## Importar bibliotecas

In [ ]:
import matplotlib.pyplot as plt

## Conectar com Google Drive

In [ ]:
# Conectar com o Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

### Variáveis de ambiente para arquivos e conversão para PDF
Alterar os caminhos de acordo com sua organização

In [ ]:
# Path para pasta de DADOS
caminho_dados = '/content/gdrive/MyDrive/DADOS/ADS/'

# Path para pasta de PDF
caminho_pdf = '/content/gdrive/MyDrive/PDF/ADS'

# Nome do notebook
nome = 'SimulaçãoFilaMG1Aula'

## Filas MG1
O sistema possui 1 servidor e uma fila de espera de tamanho infinito.<br><br>
As tarefas chegam segundo um processos de Poisson com taxa $\lambda$, ou seja, o tempo médio decorrido entre duas chegadas consecutivas (variável $T$) tem distribuição exponencial com média $\frac{1}{\lambda}$.<br><br>
O tempo de serviço para cada tarefa (variável $S$) tem distribuição genérica (qualquer) com media E_S e desvio padrão DP_S.<br><br>
A variável $R$ representa o tempo de resposta do sistema, ou seja, o tempo decorrido desde a chegada no sistema até a saída do mesmo.<br><br>
A variável $W$ representa o tempo de espera na fila, ou seja, o tempo que uma tafefa permanece esperando na fila.<br><br>
A variável $N$ representa a quantidade de tarefas dentro do sistema.<br><br>
A variável $N_{q}$ representa a quantidade de tarefas na fila.<br><br>
A variável $N_{s}$ representa a quantidade de tarefas sendo servidas.<br><br>
Os parâmetros do sistema são:
* Quantidade de servidores: $m$
* Taxa de chegada: $\lambda$
* Média do tempo de serviço: E_S
* Desvio padrão do tempo de serviço: DP_S
* Coeficiente de variação do tempo de serviço: C_S
* Quantidade de eventos simulados: $nsim$

## Classe Engine MG1
A classe **engineMG1** implementa a simulação de filas MG1.<br><br>
Argumentos do contrutor: taxa de chegadas $\lambda$, array com tempos de serviço (T_S), a quantidade de servidores (m=1) e flag para trace.<br><br>
Atributos dos objetos da classe engineMMm:
* ro - $\lambda$ * E_S
* E_T - tempo médio entre duas chegadas consecutivas
* E_W - tempo médio de espera na fila
* E_S - tempo de serviço médio
* DP_S - desvio padrão do tempo de serviço
* C_S - coeficiente de variação do tempo de serviço
* E_R - tempo de resposta médio
* E_Nq - número médio de tarefas na fila
* E_Ns - número médio de tarefas servidas
* E_N - número médio de tarefas no sistema

Objetos da classe **engineMG1** oferecem o método $simula$ que recebe como argumento o número de simulações ($nsim$), e devolvem um array com os seguintes resultados de simulação:
* EW_sim - tempo médio de espera na fila
* ES_sim - tempo de serviço médio
* ER_sim - tempo de resposta médio
* ENq_sim - número médio de tarefas na fila
* ENs_sim - número médio de tarefas servidas
* EN_sim - número médio de taferas no sistema
* U_sim - taxa de utilização do sistema

In [ ]:
import numpy as np
import scipy.stats as st
from math import factorial
from math import exp

class engineMG1:
    def __init__(self, lb, T_S, m, tr = False):
        if (lb >= m/np.mean(T_S)):
            raise ValueError('Taxa de chegada *  média do tempo de serviço deve ser menor do que 1')
        # Argumentos
        self.lb = float(lb)
        self.E_S = np.mean(T_S)
        self.DP_S = np.std(T_S)
        self.m = m
        self.trace = tr

        # Parâmetros
        self.E_T = 1 / self.lb
        self.ro = lb * self.E_S
        self.C_S = self.DP_S / self.E_S

        # Valores médios
        # valor esperado de Ns (quantidade media de tarefas servidas)
        self.E_Ns = self.ro
        # valor esperado de Nq (tamanho medio da fila)
        self.E_Nq = ((self.ro ** 2) * (1 + self.C_S ** 2)) / (2 * (1 - self.ro))
        # valor esperado de N (quantidade media de tarefas no sistema)
        self.E_N = self.E_Nq + self.E_Ns
        # valor esperado de W (tempo de espera medio na fila)
        self.E_W = (self.ro * self.E_S * (1 + self.C_S ** 2)) / (2 * (1 - self.ro))
        # valor esperado de R (tempo de resposta medio)
        self.E_R = self.E_S + self.E_W

        # Variáveis
        self.tempo = st.expon.rvs(0, self.E_T,size=1)[0]
        # Agenda é uma matriz de dimenção 1x(m+1)
        # Registra os tempos para os próximos eventos
        # Posição zero salva o tempo da próxima chegada
        # Posições 1 até m salvam os tempos para o próximo evento em cada servidor
        # Tempos para o próximo evento nos servidor são iniciados com valores muito altos
        self.Agenda = np.ones((1,self.m+1))*exp(30)
        # Tempo para a primeira chegada no sistema
        self.Agenda[0, m] = self.tempo
        # Estado do servidor é um vetor com uma posição por servidor
        # 0 = livre   1 = ocupado
        self.estado_servidor = np.zeros((1, self.m))  # todos livres
        # Tempo da ocorrência do último evento
        self.tempo_do_ultimo_evento = 0.0
        # Acumulador do tempo de serviço
        self.total_de_tempo_de_servico = 0.0
        # Acumulador do número de eventos
        self.numero_de_eventos = 1
        # Variável para salvar o número do servidor
        self.servidor = float(0)
        # Índice para tempo de serviço no array T_S
        self.indice_TS = 0

        # A divisão da variável tamanho_fila_x_tempo pela variável tempo
        # simula o tamanho médio da fila
        self.tamanho_da_fila = 0
        self.tamanho_fila_x_tempo = 0.0
        self.estado_do_sistema = 0
        self.estado_sistema_x_tempo = 0.0

    def proximo_evento(self):
        # Seleciona evento com menor valor de tempo na Agenda
        return (self.Agenda == self.Agenda.min(1, keepdims=True)).argmax(1)[0]

    def simula(self, nsim):
        if (self.trace):
            print('Agenda inicial')
            print(self.Agenda)
        while (self.numero_de_eventos < nsim):
            indice_proximo_evento = self.proximo_evento()
            # Determina tipo do próximo evento(chegada ou saída)
            if (indice_proximo_evento == self.m):
                tipo_proximo_evento = 1 # chegada
            else:
                tipo_proximo_evento = 0 # saída
                # Servidor onde ocoore a saída
                self.servidor = indice_proximo_evento
            # Recupera o tempo do próximo evento
            self.tempo = self.Agenda[0, indice_proximo_evento]

            # Atualiza os acumuladores de tempo de rodas as tarefas (área)
            self.atualiza()

            # Processa o próximo evento de acordo com o tipo
            if (tipo_proximo_evento == 1):
                self.chegada() # Chama rotina de chegada
            else:
                self.saida() # Chama rotina de saida

        # Cálculo das estatisticas da simulação
        EW_sim = self.tamanho_fila_x_tempo / self.numero_de_eventos
        ES_sim = self.total_de_tempo_de_servico / self.numero_de_eventos
        ER_sim = EW_sim + ES_sim
        ENq_sim = self.tamanho_fila_x_tempo / self.tempo
        ENs_sim = self.lb * ES_sim
        EN_sim = ENq_sim + ENs_sim
        U_sim = self.estado_sistema_x_tempo / self.tempo

        return [EW_sim, ES_sim, ER_sim, ENq_sim, ENs_sim, EN_sim, U_sim]

    def atualiza(self):
        # Calcula intervalo decorrido desde ultimo evento
        # Na primeira chamada tempo_do_ultimo_evento é igual a 0
        intervalo_do_ultimo_evento = self.tempo - self.tempo_do_ultimo_evento

        # Atualiza tempo_do_ultimo_evento
        self.tempo_do_ultimo_evento = self.tempo

        # Acumula quantidade de elementos na fila multiplicado
        # pelo tempo decorrido desde o ultimo evento
        self.tamanho_fila_x_tempo = \
            self.tamanho_fila_x_tempo + \
            self.tamanho_da_fila * intervalo_do_ultimo_evento

        # Acumula estado do servidor multiplicado
        # pelo tempo decorrido desde o ultimo evento
        self.estado_sistema_x_tempo = \
            self.estado_sistema_x_tempo + \
            self.estado_do_sistema * intervalo_do_ultimo_evento

    def chegada(self):
        if (self.trace):
            print('\nCHEGADA {:.10f}'.format(self.tempo))
            print('Agenda antes')
            print(self.Agenda)
            print('Estado dos servidores antes')
            print(self.estado_servidor)
        quantidade_servindo = np.sum(self.estado_servidor)

        if (quantidade_servindo < self.m):
            self.servidor = \
                (self.estado_servidor == self.estado_servidor.min(1, keepdims=True)).argmax(1)
            self.estado_servidor[0,self.servidor] = 1 # Servidor ocupado
            if (self.trace):
                print('Estado dos servidores depois')
                print(self.estado_servidor)
            self.estado_do_sistema = 1 # Sistema ocupado

            # Agenda a saida da tarefa que está no servidor
            
            # tempo_de_servico = st.expon.rvs(0, self.E_S,size=1)[0]
            tempo_de_servico = T_S[self.indice_TS]
            self.indice_TS = self.indice_TS + 1

            
            self.Agenda[0,self.servidor] = self.tempo + tempo_de_servico
            self.total_de_tempo_de_servico = \
            self.total_de_tempo_de_servico + tempo_de_servico

        else: # Todos servidores ocupados
            # Incrementa numero de eventos na fila
            self.tamanho_da_fila = self.tamanho_da_fila + 1

        # Agenda a proxima chegada
        self.Agenda[0,self.m] = \
            self.tempo + st.expon.rvs(0, self.E_T,size=1)[0]
        if (self.trace):
            print('Tamanho da fila = {:.0f}'.format(self.tamanho_da_fila))
        if (self.trace):
            print('Agenda depois')
            print(self.Agenda)

        # Incrementa quantidade de eventos que passaram pelo sistema
        self.numero_de_eventos = self.numero_de_eventos + 1


    def saida(self):
        if (self.trace):
            print('\nSAIDA {:.10f}'.format(self.tempo))
            print('Agenda antes')
            print(self.Agenda)
            print('Estado dos servidores antes')
            print(self.estado_servidor)
        if (self.tamanho_da_fila == 0): # Fila vazia
            # Marca o servidor como disponivel
            self.estado_servidor[0,self.servidor] = 0

            # Tratamento do proximo evento de fim de servico (muito grande)
            self.Agenda[0,self.servidor] = exp(30)
            # Atualiza quantidade servindo
            self.quantidade_servindo = np.sum(self.estado_servidor)
            if (self.quantidade_servindo == 0):
                self.estado_do_sistema = 0 # ocioso

        else: # Fila nao vazia
            # Decrementa quantidade de tarefas na fila
            self.tamanho_da_fila = self.tamanho_da_fila - 1

            # Programa o proximo evento de fim de servico
            tempo_de_servico = st.expon.rvs(0, self.E_S,size=1)[0]
            self.Agenda[0,self.servidor] = self.tempo + tempo_de_servico
            self.total_de_tempo_de_servico =\
                self.total_de_tempo_de_servico + tempo_de_servico

        if (self.trace):
            if (self.trace):
                print('Tamanho da fila = {:.0f}'.format(self.tamanho_da_fila))
            print('Agenda depois')
            print(self.Agenda)
            print('Estado dos servidores depois')
            print(self.estado_servidor)

## Exemplo
* O arquivo dados_TS_servidor.txt contém registros de tempos de serviço (em milissegundos) de 15.000 tarefas submetidas a um servidor de aplicação.
* A média dos tempos de serviço observados é de 6,269 milissegundos.
* A maior taxa de de chegada suportada para essa média de tempo de serviço é de 0,160.
* A taxa de chegada observada para as tarefas observadas foi de 0,09 tarefas por milissegundo.

In [ ]:
T_S = np.loadtxt(caminho_dados + 'dados_TS_servidor.txt', delimiter=',')
print(f"Quantidade de registros: {len(T_S):.0f}")
print(f"Média do tempo de serviço: {np.mean(T_S):.3f} ms")
print(f"Maior taxa de chegada suportada: {1/np.mean(T_S):.3f} ms")
plt.hist(T_S)
plt.show()

## Teste de simulação

### Criar um engine de simulação para um sistema MG1 com o seguintes parâmetros
* Taxa de chegada = 0.09
* Número de servidores = 1
* tr = False

In [ ]:
lb=0.09
m=1
engine = engineMG1(lb, T_S, m, tr = False)

### Simulação
* Número de simulçãoes = 15.000
* Simular
* Plotar resultados para comparação

In [ ]:
### OBSERVAR NOME DAS VARIÁVEIS RETORNADAS PARA FACIULITAR O RELATÓRIO 
nsim = 15000
EW_sim, ES_sim, ER_sim, ENq_sim, ENs_sim, EN_sim, U_sim = engine.simula(nsim)

### Relatório de simulação
Observar os valores das estatísticas simuladas e analíticas

In [ ]:
print('------------------------ Parâmetros do sistema ----------------------------------------')
print('Numero de eventos                               : {:.0f}'.format(nsim))
print('Lambda                                          : {:.4f}'.format(engine.lb))
print('Numero de servidores                            : {:.4f}'.format(engine.m))
print('Ro                                              : {:.4f}'.format(engine.ro))

print('\n---------------------------Estatísticas simuladas -----------------------------------')
print('Tempo medio na fila (simulado)                  : {:.4f}'.format(EW_sim))
print('Tempo medio de servico (simulado)               : {:.4f}'.format(ES_sim))
print('Tempo medio de resposta (simulado)              : {:.4f}'.format(ER_sim))

print('\nNumero medio de tarefas na fila (simulado)      : {:.4f}'.format(ENq_sim))
print('Numero medio de tarefas servidas (simulado)     : {:.4f}'.format(ENs_sim))
print('Numero medio de tarefas no sistema (simulado)   : {:.4f}'.format(EN_sim))

print('\nTaxa de utilizacao do sistema (simulado)        : {:.4f}'.format(U_sim))


print('\n------------------------- Estatísticas analíticas ----------------------------------')
print('Tempo medio na fila (analítico)                 : {:.4f}'.format(engine.E_W))
print('Tempo medio de servico (analítico)              : {:.4f}'.format(engine.E_S))
print('Tempo medio de resposta (analítico)             : {:.4f}'.format(engine.E_R))

print('\nNumero medio de tarefas na fila (analitico)     : {:.4f}'.format(engine.E_Nq))
print('Numero medio de tarefas servidas (analitico)    : {:.4f}'.format(engine.E_Ns))
print('Numero medio de tarefas no sistema (analitico)  : {:.4f}'.format(engine.E_N))

print('\nTaxa de utilização do sistema (analítico)       : {:.4f}'.format(engine.ro))


print('\n--------------------------------------------------------------')

# Avaliação de desempenho por simulação
## Problema
Para avaliar o desempenho de um sistema é necessário levantar dados de modo sistemático e a partir deles chegar as conclusões.  Nessa atividade vamos usar a classe de simulação **engineMG1** para levantar dados sobre um sistema com filas para avaliá-lo.<br><br>
Vamos considerar o caso em que o sistema tem 1 servidor ($m = 1$). A média do tempo de serviço obtida no levantamento de dados foi de 6.629 ms. A condição de estabilidade do sistema é $\lambda\lt \frac{1}{ES}$, ou seja, $\lambda\lt 0.160$.<br><br>
Vamos avaliar 8 valores de carga ($\lambda$): 0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15 tarefas por **milisegundo**, plotando o gráfico de tempo médio na fila, com valores teóricos calculados a partir das fórmulas e com os valores obtidos na simulação.

## Plotar gráficos do tempo médio na fila
#### Valores teóricos e simulados

### Gerar array 2D para valores teóricos e simulados
* Número de simulações = 15.000
* Valores de taxa de chegada = [0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14. 0.15 ]

Observar que os valores armazenados nos arrays 2D são posicionais. Cada coluna dos arrays vai conter os valores simulados e analícos para cada variável:
* coluna 0 - tempo médio de resposta
* coluna 1 - tempo médio de serviço
* coluna 2 - tempo médio de resposta
* coluna 3 - número médio de tarefas na fila
* coluna 4 - número médio de tarefas servidas
* coluna 5 - número médio de tarefas no sistema
* coluna 6 - taxa de utilização do sistema

In [ ]:
# Gerar array 2D para valores teóricos e simulados
m=1
nsim = 15000
simulado = []
analitico = []
LB = [0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15]
for lb in LB:
    engine = engineMG1(lb, T_S, m)
    simulado.append(engine.simula(nsim))
    analitico.append([engine.E_W, engine.E_S, engine.E_R, engine.E_Nq, engine.E_Ns, engine.E_N, engine.ro])

### Plotar gráfico do tempo médio de resposta em função dos valores de lambda

In [ ]:
# Tempo médio na fila simulado vs analítico
fig, ax = plt.subplots()  # Cria figura com um único subplot (eixo ax)

# Plotar
#   Eixo x: LB
#   eixo y: s[0]    para dados simulados
#   eixo y: a[0]    para valores analíticos
ax.plot(LB, [s[2] for s in simulado], label='simulado');  # Plotar os dados
ax.plot(LB, [a[2] for a in analitico], label='analitico');  # Plotar os dados

# Título do gráfico
plt.title('Tempo de resposta médio')

# Título do eixo x
plt.xlabel('Lambda')
plt.legend();

##### Podemos observar que a taxa de chegada deve ser menor ou igual a 0.14 ms para obter um tempo de resposta médio menor do que 42 ms na configuração atual.

In [ ]:
print('Tempo medio de resposta simulado = {:.4f} para taxa de chegadas = {:.2f}'.format(simulado[6][2], LB[6]))

## Problema
Premissas:
* Deseja-se implantar um cluster com dois servidores para melhorar a confiabilidade do sistema.
* A carga do sistema vai aumentar nos próximos meses.
* Deseja-se garantir que o tempo de resposta médio seja menor do que 50 ms.
Qual a carga máxima que o cluster pode aceitar?

In [ ]:
lb = 0.09
m = 2
ER = 0
while ((lb < m/np.mean(T_S))):
    melhor_ER = ER
    engine1 = engineMG1(lb, T_S, m, tr = False)
    simulado = engine1.simula(nsim)
    if (ER>50): break
    ER = simulado[2]
    lb = lb + 0.02

print('Tempo medio de resposta simulado = {:.4f} para taxa de chegadas = {:.2f}'.format(melhor_ER, lb))